# 01 · Đo độ phủ ghép MPD × audio features (ticket #1)

**Input** (bấm *Add Input* trong Kaggle):
- `himanshuwagh/spotify-million`, bản sao MPD có các file `mpd.slice.*.json`
- `krishsharma0413/2-million-songs-from-mpd-with-audio-features` (`extracted.sqlite`)
- Dự phòng: `rodolfofigueroa/spotify-12m-songs` (`tracks_features.csv`)

**Output:** `/kaggle/working/coverage.json`. Dán bảng kết quả vào mô tả PR của ticket #1.

Bật Internet trong Settings để clone repo. Chỉ cần CPU.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #1 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "a"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
# Tìm đường dẫn input. Thư mục mount trên Kaggle có thể khác nhau, nên tìm bằng glob.
from pathlib import Path
INPUT = Path("/kaggle/input")
slices = sorted(INPUT.rglob("mpd.slice.*.json"))
MPD_DIR = slices[0].parent if slices else None
SQLITE = next(iter(sorted(INPUT.rglob("*.sqlite")) + sorted(INPUT.rglob("*.db"))), None)
CSV_FALLBACK = next(iter(sorted(INPUT.rglob("tracks_features.csv"))), None)
print("MPD:", MPD_DIR, f"({len(slices)} slice)")
print("SQLite:", SQLITE)
print("CSV dự phòng:", CSV_FALLBACK)

In [ ]:
from src.data.coverage import list_slices, sample_slices, load_mpd_occurrences, load_features, coverage_report, format_report
N_SLICES, RANDOM_SEED = 100, 42   # 100 slice ≈ 100k playlist

chosen = sample_slices(list_slices(MPD_DIR), N_SLICES, RANDOM_SEED)
occ = load_mpd_occurrences(chosen)
print(f"{len(chosen)} slice, {len(occ):,} lượt xuất hiện, {occ['track_id'].nunique():,} bài")

In [ ]:
import json
results = {}
for name, path in [("krishsharma0413", SQLITE), ("rodolfofigueroa", CSV_FALLBACK)]:
    if path is None:
        continue
    feats = load_features(path)
    r = coverage_report(occ, feats)
    r["config"] = {"n_slices": len(chosen), "random_seed": RANDOM_SEED, "features": str(path)}
    results[name] = r
    print(f"=== {name} ===")
    print(format_report(r))
    print()
    del feats

Path("/kaggle/working/coverage.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
# Bảng markdown để dán vào PR (không cần tabulate).
rows = ["track_coverage", "occurrence_coverage", "track_coverage_usable", "occurrence_coverage_usable",
        "track_coverage_usable_among_ge5_occ", "playlists_ge10_usable_tracks"]
names = list(results)
fmt = lambda x: "—" if x is None else f"{x:.1%}"
md_lines = ["| chỉ số | " + " | ".join(names) + " |", "|---|" + "---|" * len(names)]
md_lines += ["| " + row + " | " + " | ".join(fmt(results[n][row]) for n in names) + " |" for row in rows]
md_lines += ["| n_unique_tracks / n_occurrences | " + " | ".join(f"{results[n]['n_unique_tracks']:,} / {results[n]['n_occurrences']:,}" for n in names) + " |"]
print(chr(10).join(md_lines))

## Quyết định (chạy ngày 21/11/2026)

| chỉ số | krishsharma0413 |
|---|---|
| track_coverage | 99,98% |
| occurrence_coverage | 100,00% |
| track_coverage_usable | 99,89% |
| occurrence_coverage_usable | 99,97% |
| track_coverage_usable_among_ge5_occ | 99,95% |
| playlists_ge10_usable_tracks | 96,49% |

100 slice, 100.000 playlist, 6.624.113 lượt, 686.633 bài. 0,09% bài ghép được có tempo ≤ 0.

**Giữ `krishsharma0413`**: vượt xa ngưỡng (≥ 80% lượt dùng được, ≥ 70% playlist còn ≥ 10 bài). Không cần bộ dự phòng. Xem ADR 0001.